In [7]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist

In [8]:
R=6371.00
def haversine_distance(coord1,coord2):
    lat1,long1=np.radians(coord1[:,0]),np.radians(coord1[:,1])
    lat2,long2=np.radians(coord2[:,0]),np.radians(coord2[:,1])

    dlat=lat2[:,np.newaxis]-lat1
    dlong=long2[:,np.newaxis]-long1

    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2[:, np.newaxis]) * np.sin(dlong/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    return R * c

In [9]:
df_nodes=pd.read_csv('/kaggle/input/datasets/anubhashukla/spatial-weight-matrix/nodes (1).csv')
coords=df_nodes[['Lat','Lon']].values
distance_matrix=haversine_distance(coords,coords)

In [10]:
with np.errstate(divide='ignore'):
    W=1.0/distance_matrix
    np.fill_diagonal(W,0)

W_normalized=W/W.sum(axis=1)[:,np.newaxis]
np.save('spatial_weight_matrix.npy',W_normalized)

In [11]:
df_ts=pd.read_csv('/kaggle/input/datasets/anubhashukla/spatial-weight-matrix/grid_timeseries.csv')
# Extract an available timestamp from the data to test spatial clustering
unique_timestamps = df_ts['Timestamp'].unique()
test_hour = unique_timestamps[100] if len(unique_timestamps) > 100 else unique_timestamps[0]
print(f"⌛ Running Moran's I on snapshot target hour: {test_hour}")

# Extract data rows belonging strictly to this hour
snapshot_df = df_ts[df_ts['Timestamp'] == test_hour].copy()

# Force the data rows to follow the exact sequence of your 30 nodes in nodes.csv
ordered_power = []
global_hour_mean = float(snapshot_df['Power'].mean())

for master_node in df_nodes['Node']:
    node_row = snapshot_df[snapshot_df['Node'] == master_node]
    
    if not node_row.empty:
        ordered_power.append(float(node_row['Power'].values[0]))
    else:
        # Fallback padding if a node is missing from this hour's logging stream
        ordered_power.append(global_hour_mean)

# Convert to a flat numpy array (Guaranteed to be exactly 30 elements long)
snapshot = np.array(ordered_power)

# Run the strict Moran's I mathematical formulations
x_bar = np.mean(snapshot)
z = snapshot - x_bar
N = len(snapshot) # Exactly 30
S0 = float(np.sum(W_normalized))

# Matrix math cross-multiplication: (30, 30) meets (30, 30) layout perfectly
numerator = N * np.sum(W_normalized * np.outer(z, z))
denominator = S0 * np.sum(z**2)
morans_i = numerator / denominator

print(f"Calculated Moran's I: {morans_i}. (Values > 0 confirm spatial clustering)")

⌛ Running Moran's I on snapshot target hour: 2022-01-05 04:00:00
Calculated Moran's I: 0.5849776558948933. (Values > 0 confirm spatial clustering)
